<a href="https://colab.research.google.com/github/preemalsimona05/OPG-Age-Analysis/blob/main/Dental_Age_Estimation_UNet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pycocotools.coco import COCO

DRIVE_FOLDER_PATH = '/content/drive/MyDrive/DENTAL_INTERNSHIP/OPG Age Analysis.coco-segmentation/train'
json_path = os.path.join(DRIVE_FOLDER_PATH, '_annotations.coco.json')
img_dir = DRIVE_FOLDER_PATH

coco = COCO(json_path)
filename_to_img = {img['file_name']: img for img in coco.imgs.values()}

displayed = 0
for file_name, img_info in filename_to_img.items():
    if displayed >= 3:
        break

    img_path = os.path.join(img_dir, file_name)
    raw_img = cv2.imread(img_path)
    if raw_img is None:
        continue
    raw_img = cv2.cvtColor(raw_img, cv2.COLOR_BGR2RGB)

    img_id = img_info['id']
    ann_ids = coco.getAnnIds(imgIds=img_id)
    anns = coco.loadAnns(ann_ids)

    if len(anns) == 0:
        continue

    # Create Binary Mask
    mask = np.zeros((img_info['height'], img_info['width']), dtype=np.uint8)
    for ann in anns:
        mask = np.maximum(mask, coco.annToMask(ann))

    # Calculate bounding box around ALL annotations to crop and zoom in
    y_indices, x_indices = np.where(mask > 0)
    if len(x_indices) > 0 and len(y_indices) > 0:
        x_min, x_max = np.min(x_indices), np.max(x_indices)
        y_min, y_max = np.min(y_indices), np.max(y_indices)

        # Add a 50px padding around the zoomed area
        pad = 50
        y_min = max(0, y_min - pad)
        y_max = min(img_info['height'], y_max + pad)
        x_min = max(0, x_min - pad)
        x_max = min(img_info['width'], x_max + pad)
    else:
        continue

    # Crop the raw image and mask to the zoomed area
    cropped_img = raw_img[y_min:y_max, x_min:x_max]
    cropped_mask = mask[y_min:y_max, x_min:x_max]

    # Draw thick red contours on the cropped view
    contours, _ = cv2.findContours(cropped_mask, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)
    overlay_crop = cropped_img.copy()
    cv2.drawContours(overlay_crop, contours, -1, (255, 0, 0), 3) # Red outlines

    # Plot
    fig, ax = plt.subplots(1, 3, figsize=(15, 5))

    ax[0].imshow(cropped_img)
    ax[0].set_title("1. Zoomed Tooth Area")
    ax[0].axis('off')

    ax[1].imshow(overlay_crop)
    ax[1].set_title("2. Annotation Outlines (Red)")
    ax[1].axis('off')

    ax[2].imshow(cropped_mask, cmap='gray')
    ax[2].set_title("3. Zoomed Binary Mask")
    ax[2].axis('off')

    plt.tight_layout()
    plt.show()

    displayed += 1

In [ ]:
# Display ALL images in the dataset
for file_name, img_info in filename_to_img.items():
    img_path = os.path.join(img_dir, file_name)
    raw_img = cv2.imread(img_path)
    if raw_img is None:
        continue
    raw_img = cv2.cvtColor(raw_img, cv2.COLOR_BGR2RGB)

    img_id = img_info['id']
    ann_ids = coco.getAnnIds(imgIds=img_id)
    anns = coco.loadAnns(ann_ids)

    if len(anns) == 0:
        print(f"Skipping {file_name}: No annotations found.")
        continue

    mask = np.zeros((img_info['height'], img_info['width']), dtype=np.uint8)
    for ann in anns:
        mask = np.maximum(mask, coco.annToMask(ann))

    y_indices, x_indices = np.where(mask > 0)
    if len(x_indices) > 0 and len(y_indices) > 0:
        x_min, x_max = np.min(x_indices), np.max(x_indices)
        y_min, y_max = np.min(y_indices), np.max(y_indices)

        pad = 50
        y_min = max(0, y_min - pad)
        y_max = min(img_info['height'], y_max + pad)
        x_min = max(0, x_min - pad)
        x_max = min(img_info['width'], x_max + pad)

        cropped_img = raw_img[y_min:y_max, x_min:x_max]
        cropped_mask = mask[y_min:y_max, x_min:x_max]

        contours, _ = cv2.findContours(cropped_mask, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)
        overlay_crop = cropped_img.copy()
        cv2.drawContours(overlay_crop, contours, -1, (255, 0, 0), 3)

        fig, ax = plt.subplots(1, 3, figsize=(15, 5))
        ax[0].imshow(cropped_img)
        ax[0].set_title(f"1. Zoomed: {file_name}")
        ax[0].axis('off')

        ax[1].imshow(overlay_crop)
        ax[1].set_title(f"2. Annotations ({len(anns)} polygons)")
        ax[1].axis('off')

        ax[2].imshow(cropped_mask, cmap='gray')
        ax[2].set_title("3. Binary Mask")
        ax[2].axis('off')

        plt.tight_layout()
        plt.show()

In [ ]:
import os
import random
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms.functional as TF
from pycocotools.coco import COCO

# -------------------------------------------------------------
# 1. COMBINED DICE + BCE LOSS FUNCTION
# -------------------------------------------------------------
class DiceBCELoss(nn.Module):
    def __init__(self):
        super(DiceBCELoss, self).__init__()
        self.bce = nn.BCELoss()

    def forward(self, inputs, targets, smooth=1e-6):
        bce_loss = self.bce(inputs, targets)

        inputs_flat = inputs.view(-1)
        targets_flat = targets.view(-1)

        intersection = (inputs_flat * targets_flat).sum()
        dice_loss = 1 - (2. * intersection + smooth) / (inputs_flat.sum() + targets_flat.sum() + smooth)

        return bce_loss + dice_loss

# -------------------------------------------------------------
# 2. DATASET DEFINITION
# -------------------------------------------------------------
class DentalAugmentedDataset(Dataset):
    def __init__(self, json_path, img_dir, img_size=(256, 256)):
        self.coco = COCO(json_path)
        self.img_dir = img_dir
        self.img_size = img_size
        self.filename_to_img = {img['file_name']: img for img in self.coco.imgs.values()}
        self.file_list = list(self.filename_to_img.keys())

    def __len__(self):
        return len(self.file_list)

    def __getitem__(self, index):
        file_name = self.file_list[index]
        img_info = self.filename_to_img[file_name]

        img_path = os.path.join(self.img_dir, file_name)
        image = Image.open(img_path).convert("L")

        img_id = img_info['id']
        ann_ids = self.coco.getAnnIds(imgIds=img_id)
        anns = self.coco.loadAnns(ann_ids)
        mask_np = np.zeros((img_info['height'], img_info['width']), dtype=np.float32)
        for ann in anns:
            mask_np = np.maximum(mask_np, self.coco.annToMask(ann))
        mask = Image.fromarray((mask_np * 255).astype(np.uint8))

        image = TF.resize(image, self.img_size)
        mask = TF.resize(mask, self.img_size)

        image_tensor = TF.to_tensor(image)
        mask_tensor = TF.to_tensor(mask)
        mask_tensor = (mask_tensor > 0.5).float()

        return image_tensor, mask_tensor, file_name

# -------------------------------------------------------------
# 3. U-NET ARCHITECTURE
# -------------------------------------------------------------
class UNet(nn.Module):
    def __init__(self):
        super(UNet, self).__init__()
        self.enc1 = nn.Sequential(nn.Conv2d(1, 16, 3, padding=1), nn.ReLU())
        self.pool1 = nn.MaxPool2d(2, 2)
        self.enc2 = nn.Sequential(nn.Conv2d(16, 32, 3, padding=1), nn.ReLU())
        self.pool2 = nn.MaxPool2d(2, 2)

        self.bottleneck = nn.Sequential(nn.Conv2d(32, 64, 3, padding=1), nn.ReLU())

        self.up1 = nn.ConvTranspose2d(64, 32, 2, stride=2)
        self.dec1 = nn.Sequential(nn.Conv2d(32, 32, 3, padding=1), nn.ReLU())
        self.up2 = nn.ConvTranspose2d(32, 16, 2, stride=2)
        self.dec2 = nn.Sequential(nn.Conv2d(16, 16, 3, padding=1), nn.ReLU())

        self.final = nn.Conv2d(16, 1, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool1(e1))
        b = self.bottleneck(self.pool2(e2))
        d1 = self.dec1(self.up1(b))
        d2 = self.dec2(self.up2(d1))
        return torch.sigmoid(self.final(d2))

# -------------------------------------------------------------
# 4. TRAIN MODEL
# -------------------------------------------------------------
DRIVE_FOLDER_PATH = '/content/drive/MyDrive/DENTAL_INTERNSHIP/OPG Age Analysis.coco-segmentation/train'
json_path = os.path.join(DRIVE_FOLDER_PATH, '_annotations.coco.json')

dataset = DentalAugmentedDataset(json_path, DRIVE_FOLDER_PATH)
dataloader = DataLoader(dataset, batch_size=2, shuffle=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = UNet().to(device)
criterion = DiceBCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

print(f"Training on device: {device}")
model.train()
for epoch in range(25):
    epoch_loss = 0.0
    for images, masks, _ in dataloader:
        images, masks = images.to(device), masks.to(device)
        optimizer.zero_grad()
        preds = model(images)
        loss = criterion(preds, masks)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()

    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"Epoch [{epoch+1}/25] - Loss: {epoch_loss/len(dataloader):.4f}")

# -------------------------------------------------------------
# 5. PREDICT & VISUALIZE ALL 10 IMAGES
# -------------------------------------------------------------
model.eval()

for idx in range(len(dataset)):
    test_img, test_mask, file_name = dataset[idx]

    with torch.no_grad():
        input_tensor = test_img.unsqueeze(0).to(device)
        raw_pred = model(input_tensor).cpu().squeeze().numpy()
        pred_binary = (raw_pred > 0.25).astype(np.uint8)

    orig_img = test_img.squeeze().numpy()
    ground_truth = test_mask.squeeze().numpy()

    # Overlay predicted mask in red
    overlay = np.stack([orig_img]*3, axis=-1)
    overlay[pred_binary == 1] = [1.0, 0.2, 0.2]

    # Plot each of the 10 images
    fig, ax = plt.subplots(1, 3, figsize=(15, 5))

    ax[0].imshow(orig_img, cmap='gray')
    ax[0].set_title(f"Image {idx+1}/10: Raw Input")
    ax[0].axis('off')

    ax[1].imshow(ground_truth, cmap='gray')
    ax[1].set_title("Ground Truth Mask")
    ax[1].axis('off')

    ax[2].imshow(overlay)
    ax[2].set_title("AI Predicted Overlay (Red)")
    ax[2].axis('off')

    plt.tight_layout()
    plt.show()

In [ ]:
import os
import random
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms.functional as TF
from pycocotools.coco import COCO

# -------------------------------------------------------------
# 1. STABLE DICE + BCE COMBINED LOSS
# -------------------------------------------------------------
class DiceBCELoss(nn.Module):
    def __init__(self, weight_bce=1.0, weight_dice=1.0):
        super(DiceBCELoss, self).__init__()
        self.bce = nn.BCELoss()
        self.w_bce = weight_bce
        self.w_dice = weight_dice

    def forward(self, inputs, targets, smooth=1e-5):
        bce_loss = self.bce(inputs, targets)

        inputs_flat = inputs.view(-1)
        targets_flat = targets.view(-1)

        intersection = (inputs_flat * targets_flat).sum()
        dice_score = (2. * intersection + smooth) / (inputs_flat.sum() + targets_flat.sum() + smooth)
        dice_loss = 1.0 - dice_score

        return (self.w_bce * bce_loss) + (self.w_dice * dice_loss)

# -------------------------------------------------------------
# 2. AUGMENTED DATASET (Oversampled)
# -------------------------------------------------------------
class DentalAugmentedDataset(Dataset):
    def __init__(self, json_path, img_dir, img_size=(256, 256)):
        self.coco = COCO(json_path)
        self.img_dir = img_dir
        self.img_size = img_size
        self.filename_to_img = {img['file_name']: img for img in self.coco.imgs.values()}
        self.file_list = list(self.filename_to_img.keys())

    def __len__(self):
        # 10 images x 20 augmented variations per epoch = 200 samples
        return len(self.file_list) * 20

    def __getitem__(self, index):
        real_idx = index % len(self.file_list)
        file_name = self.file_list[real_idx]
        img_info = self.filename_to_img[file_name]

        img_path = os.path.join(self.img_dir, file_name)
        image = Image.open(img_path).convert("L")

        img_id = img_info['id']
        ann_ids = self.coco.getAnnIds(imgIds=img_id)
        anns = self.coco.loadAnns(ann_ids)
        mask_np = np.zeros((img_info['height'], img_info['width']), dtype=np.float32)
        for ann in anns:
            mask_np = np.maximum(mask_np, self.coco.annToMask(ann))
        mask = Image.fromarray((mask_np * 255).astype(np.uint8))

        image = TF.resize(image, self.img_size)
        mask = TF.resize(mask, self.img_size)

        if random.random() > 0.5:
            image = TF.hflip(image)
            mask = TF.hflip(mask)

        angle = random.uniform(-8, 8)
        image = TF.rotate(image, angle)
        mask = TF.rotate(mask, angle)

        image_tensor = TF.to_tensor(image)
        mask_tensor = TF.to_tensor(mask)
        mask_tensor = (mask_tensor > 0.5).float()

        return image_tensor, mask_tensor, file_name

# -------------------------------------------------------------
# 3. U-NET MODEL
# -------------------------------------------------------------
class UNet(nn.Module):
    def __init__(self):
        super(UNet, self).__init__()
        self.enc1 = nn.Sequential(nn.Conv2d(1, 16, 3, padding=1), nn.BatchNorm2d(16), nn.ReLU())
        self.pool1 = nn.MaxPool2d(2, 2)
        self.enc2 = nn.Sequential(nn.Conv2d(16, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU())
        self.pool2 = nn.MaxPool2d(2, 2)

        self.bottleneck = nn.Sequential(nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU())

        self.up1 = nn.ConvTranspose2d(64, 32, 2, stride=2)
        self.dec1 = nn.Sequential(nn.Conv2d(32, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU())
        self.up2 = nn.ConvTranspose2d(32, 16, 2, stride=2)
        self.dec2 = nn.Sequential(nn.Conv2d(16, 16, 3, padding=1), nn.BatchNorm2d(16), nn.ReLU())

        self.final = nn.Conv2d(16, 1, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool1(e1))
        b = self.bottleneck(self.pool2(e2))
        d1 = self.dec1(self.up1(b))
        d2 = self.dec2(self.up2(d1))
        return torch.sigmoid(self.final(d2))

# -------------------------------------------------------------
# 4. RUN TRAINING (100 Epochs)
# -------------------------------------------------------------
DRIVE_FOLDER_PATH = '/content/drive/MyDrive/DENTAL_INTERNSHIP/OPG Age Analysis.coco-segmentation/train'
json_path = os.path.join(DRIVE_FOLDER_PATH, '_annotations.coco.json')

dataset = DentalAugmentedDataset(json_path, DRIVE_FOLDER_PATH)
dataloader = DataLoader(dataset, batch_size=8, shuffle=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = UNet().to(device)
criterion = DiceBCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.0005)

print(f"Training on device: {device} for 100 Epochs...")
model.train()
for epoch in range(100):
    epoch_loss = 0.0
    for images, masks, _ in dataloader:
        images, masks = images.to(device), masks.to(device)
        optimizer.zero_grad()
        preds = model(images)
        loss = criterion(preds, masks)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()

    if (epoch + 1) % 10 == 0 or epoch == 0:
        print(f"Epoch [{epoch+1}/100] - Loss: {epoch_loss/len(dataloader):.4f}")

# -------------------------------------------------------------
# 5. DISPLAY PREDICTIONS FOR ALL 10 IMAGES
# -------------------------------------------------------------
model.eval()

# Load un-augmented dataset for evaluation
eval_dataset = DentalAugmentedDataset(json_path, DRIVE_FOLDER_PATH)
eval_dataset.file_list = list(eval_dataset.filename_to_img.keys()) # reset to exact 10 files

for idx in range(10):
    file_name = eval_dataset.file_list[idx]
    img_info = eval_dataset.filename_to_img[file_name]

    # Load original image
    img_path = os.path.join(DRIVE_FOLDER_PATH, file_name)
    raw_pil = Image.open(img_path).convert("L")
    resized_pil = TF.resize(raw_pil, (256, 256))
    input_tensor = TF.to_tensor(resized_pil).unsqueeze(0).to(device)

    # Load original mask
    img_id = img_info['id']
    ann_ids = eval_dataset.coco.getAnnIds(imgIds=img_id)
    anns = eval_dataset.coco.loadAnns(ann_ids)
    mask_np = np.zeros((img_info['height'], img_info['width']), dtype=np.float32)
    for ann in anns:
        mask_np = np.maximum(mask_np, eval_dataset.coco.annToMask(ann))
    mask_pil = TF.resize(Image.fromarray((mask_np * 255).astype(np.uint8)), (256, 256))
    ground_truth = np.array(mask_pil) / 255.0

    with torch.no_grad():
        raw_pred = model(input_tensor).cpu().squeeze().numpy()
        pred_binary = (raw_pred > 0.3).astype(np.uint8)

    orig_img = np.array(resized_pil) / 255.0

    # Overlay predicted mask in red
    overlay = np.stack([orig_img]*3, axis=-1)
    overlay[pred_binary == 1] = [1.0, 0.2, 0.2]

    # Plot
    fig, ax = plt.subplots(1, 3, figsize=(15, 5))

    ax[0].imshow(orig_img, cmap='gray')
    ax[0].set_title(f"Image {idx+1}/10: Raw Input")
    ax[0].axis('off')

    ax[1].imshow(ground_truth, cmap='gray')
    ax[1].set_title("Ground Truth Mask")
    ax[1].axis('off')

    ax[2].imshow(overlay)
    ax[2].set_title("AI Predicted Overlay (Red)")
    ax[2].axis('off')

    plt.tight_layout()
    plt.show()

In [ ]:
import os
import json
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import torchvision.transforms as T
import matplotlib.pyplot as plt
from pycocotools.coco import COCO

# ==========================================
# 1. PATH CONFIGURATION
# ==========================================
DRIVE_FOLDER_PATH = '/content/drive/MyDrive/DENTAL_INTERNSHIP/OPG Age Analysis.coco-segmentation/train'
json_path = os.path.join(DRIVE_FOLDER_PATH, '_annotations.coco.json')
img_dir = DRIVE_FOLDER_PATH

assert os.path.exists(json_path), f"Error: Cannot find annotation file at {json_path}"
assert os.path.exists(img_dir), f"Error: Cannot find image directory at {img_dir}"


# ==========================================
# 2. MULTI-CLASS DATASET CLASS
# ==========================================
class DentalMultiClassDataset(Dataset):
    def __init__(self, annotation_file, img_dir, transform=None):
        self.coco = COCO(annotation_file)
        self.ids = list(self.coco.imgs.keys())
        self.img_dir = img_dir
        self.transform = transform

        # Category IDs extracted directly from your COCO JSON file
        self.pulp_cat_ids = [1, 3, 5, 7, 9]        # 34 Pulp, 35 Pulp, etc.
        self.tooth_cat_ids = [2, 4, 6, 8, 10, 11]  # 34 Tooth, 35 Tooth, etc.

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, index):
        img_id = self.ids[index]
        ann_ids = self.coco.getAnnIds(imgIds=img_id)
        anns = self.coco.loadAnns(ann_ids)

        img_info = self.coco.loadImgs(img_id)[0]
        img_full_path = os.path.join(self.img_dir, img_info['file_name'])

        image = Image.open(img_full_path).convert("RGB")
        h, w = img_info['height'], img_info['width']

        mask = np.zeros((2, h, w), dtype=np.float32)

        for ann in anns:
            cat_id = ann['category_id']
            ann_mask = self.coco.annToMask(ann)

            if cat_id in self.tooth_cat_ids:
                mask[0] = np.maximum(mask[0], ann_mask)
            elif cat_id in self.pulp_cat_ids:
                mask[1] = np.maximum(mask[1], ann_mask)

        if self.transform:
            image = self.transform(image)
            mask = torch.from_numpy(mask)
            mask = T.functional.resize(mask, [512, 512], interpolation=T.InterpolationMode.NEAREST)

        return image, mask


# ==========================================
# 3. COMBINED DICE + BCE LOSS
# ==========================================
class DiceBCELoss(nn.Module):
    def __init__(self, weight=None, size_average=True):
        super(DiceBCELoss, self).__init__()
        self.bce = nn.BCEWithLogitsLoss()

    def forward(self, inputs, targets, smooth=1):
        bce_loss = self.bce(inputs, targets)

        inputs = torch.sigmoid(inputs)
        inputs = inputs.view(-1)
        targets = targets.view(-1)

        intersection = (inputs * targets).sum()
        dice_loss = 1 - (2. * intersection + smooth) / (inputs.sum() + targets.sum() + smooth)

        return bce_loss + dice_loss


# ==========================================
# 4. U-NET ARCHITECTURE (out_channels=2)
# ==========================================
class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.conv(x)

class UNetMultiClass(nn.Module):
    def __init__(self, in_channels=3, out_channels=2):
        super().__init__()
        self.inc = DoubleConv(in_channels, 64)
        self.down1 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(64, 128))
        self.down2 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(128, 256))

        self.up1 = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.conv_up1 = DoubleConv(256, 128)
        self.up2 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.conv_up2 = DoubleConv(128, 64)

        self.outc = nn.Conv2d(64, out_channels, 1)

    def forward(self, x):
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)

        x = self.up1(x3)
        x = torch.cat([x, x2], dim=1)
        x = self.conv_up1(x)

        x = self.up2(x)
        x = torch.cat([x, x1], dim=1)
        x = self.conv_up2(x)

        return self.outc(x)


# ==========================================
# 5. INITIALIZATION & TRAINING
# ==========================================
transform = T.Compose([
    T.Resize((512, 512)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

dataset = DentalMultiClassDataset(
    annotation_file=json_path,
    img_dir=img_dir,
    transform=transform
)

data_loader = DataLoader(dataset, batch_size=2, shuffle=False)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = UNetMultiClass(in_channels=3, out_channels=2).to(device)
criterion = DiceBCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.0003)

epochs = 60
print(f"Starting Multi-Class Training on device: {device}")

model.train()
for epoch in range(epochs):
    running_loss = 0.0
    for images, masks in data_loader:
        images = images.to(device)
        masks = masks.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, masks)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    if (epoch + 1) % 10 == 0:
        print(f"Epoch [{epoch+1}/{epochs}] - Loss: {running_loss/len(data_loader):.4f}")


# ==========================================
# 6. VISUALIZATION & SAVING FOR ALL IMAGES
# ==========================================
os.makedirs("results", exist_ok=True)
print("\nGenerating Predictions for ALL Images...")

model.eval()
img_counter = 1

with torch.no_grad():
    for images, masks in data_loader:
        images = images.to(device)
        outputs = torch.sigmoid(model(images))

        for i in range(len(images)):
            # Unnormalize image for plotting
            img_np = images[i].cpu().numpy().transpose(1, 2, 0)
            img_np = img_np * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])
            img_np = np.clip(img_np, 0, 1)

            pred = outputs[i].cpu().numpy()

            # Probability thresholds
            tooth_mask = pred[0] > 0.3
            pulp_mask  = pred[1] > 0.3

            # Create RGB Overlay (Blue = Tooth | Red = Pulp)
            overlay = img_np.copy()
            overlay[tooth_mask] = [0.2, 0.4, 0.9]  # Blue Tooth
            overlay[pulp_mask]  = [1.0, 0.2, 0.2]  # Red Pulp

            # Plot
            fig, ax = plt.subplots(1, 2, figsize=(12, 6))
            ax[0].imshow(img_np)
            ax[0].set_title(f"Radiograph {img_counter}")
            ax[0].axis('off')

            ax[1].imshow(overlay)
            ax[1].set_title("Overlay (Blue = Tooth | Red = Pulp)")
            ax[1].axis('off')

            # Save output image directly into local results/ folder
            save_path = f"results/prediction_opg_{img_counter}.png"
            plt.tight_layout()
            plt.savefig(save_path, bbox_inches='tight')
            plt.show()

            print(f"Saved: {save_path}")
            img_counter += 1